In [ ]:
# notebooks/01_data_download.ipynb or src/data_loader.py

import GEOparse
import pandas as pd
import numpy as np
import os

class GEODataLoader:
    """Download and parse GEO RNA-Seq data"""
    
    def __init__(self, geo_id='GSE57148'):
        self.geo_id = geo_id
        self.gse = None
        
    def download_data(self, destdir='./data/raw'):
        """Download GEO dataset"""
        print(f"Downloading {self.geo_id}...")
        self.gse = GEOparse.get_GEO(geo=self.geo_id, destdir=destdir)
        print(f"Dataset downloaded successfully!")
        return self.gse
    
    def extract_expression_matrix(self):
        """Extract gene expression matrix from GEO"""
        # Get expression data from first platform
        platform_name = list(self.gse.gpls.keys())[0]
        
        # Initialize lists to store data
        expression_data = []
        sample_names = []
        
        # Extract data from each sample
        for gsm_name, gsm in self.gse.gsms.items():
            sample_names.append(gsm_name)
            expression_data.append(gsm.table['VALUE'])
        
        # Create DataFrame
        gene_ids = self.gse.gsms[sample_names[0]].table['ID_REF']
        expr_df = pd.DataFrame(expression_data).T
        expr_df.columns = sample_names
        expr_df.index = gene_ids
        
        return expr_df
    
    def extract_metadata(self):
        """Extract sample metadata (disease status, demographics)"""
        metadata = []
        
        for gsm_name, gsm in self.gse.gsms.items():
            sample_info = {
                'sample_id': gsm_name,
                'title': gsm.metadata.get('title', [''])[0],
                'characteristics': gsm.metadata.get('characteristics_ch1', [])
            }
            
            # Parse characteristics for disease status
            for char in sample_info['characteristics']:
                if 'copd' in char.lower() or 'disease' in char.lower():
                    if 'normal' in char.lower() or 'control' in char.lower():
                        sample_info['disease_status'] = 'Normal'
                    else:
                        sample_info['disease_status'] = 'COPD'
                        
            metadata.append(sample_info)
        
        return pd.DataFrame(metadata)

# Usage
loader = GEODataLoader('GSE57148')
gse_data = loader.download_data()
expression_matrix = loader.extract_expression_matrix()
metadata = loader.extract_metadata()

# Save processed data
expression_matrix.to_csv('./data/raw/expression_matrix.csv')
metadata.to_csv('./data/raw/metadata.csv', index=False)
print(f"Expression matrix shape: {expression_matrix.shape}")
print(f"Metadata shape: {metadata.shape}")
